# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook demonstrates how to load, overview, and explore the FAIR² dataset using the `mlcroissant` library, referencing fields and entities by their `@id` for full reproducibility and traceability.

### Dataset Source
The dataset is described by a [Croissant schema](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json).


In [ ]:
# Ensure that `mlcroissant` is installed in the environment
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and available record sets using `mlcroissant`. The metadata object enables exploring dataset structure and documentation.


In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # Metadata is a single object

print(f"Dataset Name: {metadata.name if hasattr(metadata, 'name') else ''}")
print(f"Identifier (DOI): {metadata.identifier if hasattr(metadata, 'identifier') else ''}")
print(f"Description: {metadata.description if hasattr(metadata, 'description') else ''}\n")
if hasattr(metadata, 'keywords'):
    print(f"Keywords: {metadata.keywords}")
if hasattr(metadata, 'version'):
    print(f"Version: {metadata.version}")
if hasattr(metadata, 'license'):
    print(f"License: {metadata.license}")

## 2. Data Overview
Review all available record sets and their fields, referencing them by their `@id`.

**Note:** The entire dataset may have one or more record sets. For each record set, we list its `@id`, name, and all contained field `@id`s. This allows deterministic extraction and analysis of data in the next steps.

In [ ]:
# List available record sets and their fields by @id
if not hasattr(metadata, 'record_sets'):
    record_sets = [r for r in metadata.recordSet] if hasattr(metadata, 'recordSet') else []
else:
    record_sets = list(metadata.record_sets.values())

print('Record sets in the dataset:')
all_record_set_ids = []
for rs in record_sets:
    rs_id = getattr(rs, '@id', None) or getattr(rs, 'id', None)
    all_record_set_ids.append(rs_id)
    rs_name = getattr(rs, 'name', '')
    print(f'- RecordSet @id: {rs_id}')
    print(f'  Name: {rs_name}')
    if hasattr(rs, 'fields'):
        field_list = rs.fields.values() if hasattr(rs.fields, 'values') else rs.fields
    elif hasattr(rs, 'field'):
        # Different naming in croissant versions
        field_list = rs.field
    else:
        field_list = []
    print(f'  Fields @id:')
    for fld in field_list:
        f_id = getattr(fld, '@id', None) or getattr(fld, 'id', None)
        f_name = getattr(fld, 'name', '')
        print(f'    - {f_id}: {f_name}')
    print('')

## 3. Data Extraction
Extract records from one or more record sets using their `@id`. We load them into pandas DataFrames for further manipulation, using the `@id` string as the DataFrame key.

Choose the main record set (by `@id`). Substitute into the extraction logic below as shown.

In [ ]:
# List all record set @ids (from above).
# For this dataset, we often have only one primary table; modify this list if you determine more.
RECORD_SET_IDS = [
    # Example: 'https://api.app.sen.science/frontiers/7862866/record_set/clinicopathological_records',
    # Please update RECORD_SET_IDS after inspecting output above.
    # Use the exact @id values for record sets here:
]

# If you know the record set @id (e.g. output above shows it is 'https://api.app.sen.science/frontiers/7862866/63f8867d-457f-4e17-873b-2dd624b86a6d'), add it in, e.g.:
# RECORD_SET_IDS = ['https://api.app.sen.science/frontiers/7862866/63f8867d-457f-4e17-873b-2dd624b86a6d']
# For this example, let's programmatically extract @ids again in case of schema differences
if not RECORD_SET_IDS:
    RECORD_SET_IDS = []
    for rs in record_sets:
        rs_id = getattr(rs, '@id', None) or getattr(rs, 'id', None)
        if rs_id is not None:
            RECORD_SET_IDS.append(rs_id)
    print(f"Detected RECORD_SET_IDS: {RECORD_SET_IDS}")

# Now load each record set into dataframe
dataframes = {}
for record_set_id in RECORD_SET_IDS:
    rows = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(rows)
    dataframes[record_set_id] = df

# Display columns of first record set
if RECORD_SET_IDS:
    main_record_set = RECORD_SET_IDS[0]
    print(f"Columns in record set {main_record_set}:\n", dataframes[main_record_set].columns.tolist())
    display(dataframes[main_record_set].head())
else:
    print('No record sets detected for extraction.')

## 4. Exploratory Data Analysis (EDA)
This section demonstrates EDA steps: filtering, normalizing, and grouping by field `@id`. Update `numeric_field_id` and `group_field_id` below based on inspected column names (which correspond to field `@id`s, not semantic names).

- Filtering to exclude low/aberrant values
- Normalizing numeric values
- Grouping by categorical attribute (e.g. sex, cancer type)


In [ ]:
# Example values for numeric field @id and group field @id
# Examine the above step's DataFrame columns for the exact @id strings for your chosen variables.

# For illustration, suppose the table includes an age field and a sex field such as:
# numeric_field_id = 'https://api.app.sen.science/frontiers/7862866/field/age'  # modify as needed
# group_field_id = 'https://api.app.sen.science/frontiers/7862866/field/sex'  # modify as needed

if RECORD_SET_IDS:
    df = dataframes[main_record_set]
    # Pick the first column that is numeric as an example
    numeric_field_id = None
    group_field_id = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break
    # Pick a non-numeric for grouping
    for col in df.columns:
        if not pd.api.types.is_numeric_dtype(df[col]):
            group_field_id = col
            break

    if numeric_field_id is not None:
        threshold = df[numeric_field_id].mean() if pd.notnull(df[numeric_field_id].mean()) else 0
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f'Filtered records with {numeric_field_id} > {threshold:.2f}:')
        print(filtered_df.head())

        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, norm_col]].head())

        if group_field_id is not None and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[[numeric_field_id]].mean().reset_index()
            print(f"Grouped data by {group_field_id} (mean {numeric_field_id}):")
            print(grouped_df)
    else:
        print('No numeric field detected in extracted table.')
else:
    print('No dataframes loaded for EDA.')

## 5. Visualization
Visualize the distribution of a chosen numeric field and the grouping variable using matplotlib and seaborn. These graphs help reveal structure in the dataset fields by `@id`.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if RECORD_SET_IDS and numeric_field_id is not None:
    fig, axs = plt.subplots(1, 2, figsize=(12,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, ax=axs[0], color='steelblue')
    axs[0].set_title(f"Distribution of {numeric_field_id}")
    axs[0].set_xlabel(numeric_field_id)

    # Boxplot of numeric by group field (if available)
    if group_field_id is not None and group_field_id in df.columns:
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id], ax=axs[1])
        axs[1].set_title(f"{numeric_field_id} by {group_field_id}")
        axs[1].set_xlabel(group_field_id)
        axs[1].set_ylabel(numeric_field_id)
    plt.tight_layout()
    plt.show()
else:
    print('No numeric field or data available for visualization.')

## 6. Conclusion
In this notebook, we loaded the FAIR² colorectal cancer dataset using the `mlcroissant` library, explored its structure and record sets via `@id`, and performed basic exploratory data analysis and plotting. By referencing fields and record sets by `@id`, all analyses are fully reproducible against the formal schema.

To extend this analysis, map clinical interpretations to `@id`s for fields, or consult the Croissant schema at the source URL for semantic context.
